In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.dates as mdates


In [ ]:
def line_fit(X,Y):
    n = len(X)
    P, residuals, rank, singular_values, rcond= np.polyfit(X,Y,1, full=True)
    dx = np.max(X) - np.min(X)
    X_new = np.array([np.min(X)-dx*0.2, np.max(X)+dx*0.2])
    Y_new = np.polyval(P, X_new)
    rmse = np.sqrt(residuals[0] / n)
    return X_new, Y_new, P, rmse

In [ ]:
def exponential_func(x, a, b, c):
    return a * np.exp(b * x) + c

def exp_fit(X,Y, P0=np.array([1, 1e-6, 1])):
    """returns  X_new, Y_new, popt, pcov"""
    popt, pcov = curve_fit(exponential_func, X,Y, p0=P0)
    dx = np.max(X) - np.min(X)
    X_new = np.linspace( np.min(X)-dx*0.2, np.max(X)+dx*0.2, 100)    
    Y_new = exponential_func(X_new, *popt)
    return  X_new, Y_new, popt, pcov

In [ ]:
data_old = pd.read_excel('20251126_DepthLaurenPaper/CoralDepthData.xlsx')
data_old.rename(columns={data_old.columns[0]:'id', data_old.columns[1]:'d13c', data_old.columns[2]:'d18o'}, inplace=True)
data_old['path'] = [x[1] for x in data_old['id'].str.split(' ')]
data_old['idx'] = [x[-1][0] for x in data_old['id'].str.split(' ')]
depths = np.array([-105, -95, -68, -58, -48, -35, -15]) - 60
#depths = np.array([-105, -95, -73, -63, -53, -42, -32.5]) - 60
paths = np.unique(data_old['path'].values)
path_depth = dict(zip(paths,depths))
data_old['depth'] = data_old['path'].map(path_depth)
data_old

In [ ]:
data = pd.read_excel("2024-depth-ABCDE_memo-edited.xlsx")
data['path'] = [x[0][0] for x in data['Identifier 1'].str.split(' ')]
data['idx'] = [int(str.replace(x[1],'A','')) for x in data['Identifier 1'].str.split(' ')]
data.loc[(data['path'] == 'C')&(data['idx']==14), 'd13c'] = np.nan
offset = 60
paths = data['path'].dropna().unique()
depths = np.array([-100, -94, -70, -58.5, -48])-offset
#depth_dict = {'A':-100, 'B':-94, 'C':-70, 'D':-58.5, 'E':-48}
depth_dict = dict(zip(paths, depths))
data['depth'] = data['path'].map(depth_dict) 
data.iloc[70:80]

In [ ]:
paths = data['path'].unique().astype(str)

fig, ax = plt.subplots(1, figsize=(8,5))

for p in paths:

    data_iter = data[data['path'] == p]
    ax.plot(data_iter['idx'] , data_iter['d18o'] ,".-", label = "Depth "+p)
ax.legend()
ax.set_title(r"$\delta18$O")
fig.savefig("d18O.pdf")

In [ ]:
paths = data['path'].unique().astype(str)

fig, ax = plt.subplots(1, figsize=(8,5))

for p in paths:

    data_iter = data[data['path'] == p]
    ax.plot(data_iter['idx'] , data_iter['d13c'] ,".-", label = "Depth "+p)
ax.legend()
ax.grid()

ax.set_title(r"$\delta13$C")
fig.savefig("d13C.pdf")

In [ ]:
data_ = data[data['path'] != 'E']

depths = data_['depth'].unique()
paths = data_['path'].unique()
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap(depths/depths.min()-0.1)

fig, ax = plt.subplots(1, figsize=(6, 6))

for i,p in enumerate(paths):
    
    data_iter = data[data['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'],data_iter['depth'], '.', color=c)
    ax.errorbar(x=data_iter['d13c'].mean(), 
                y=np.unique(data_iter['depth']), yerr=0, fmt='xr',
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, depth_dict[p]-2, p, color=c)
x = data_.groupby('path')['d13c'].mean().values
y = depths
X, Y, P, rmse= line_fit(x,y)

ax.plot(X,Y, color='r', label=f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+f'RMSE={rmse:.3f}')

ax.legend()
ax.set_ylim(-180, -90) 
ax.set_xlim(-3.5, -1)
ax.set_yticks(np.arange(-180, -90, 10))
ax.grid()
ax.set_ylabel('Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')

fig.savefig('depth_d13c.pdf')

In [ ]:
depths = data['depth'].unique()
paths = data['path'].unique()
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap(depths/depths.min()-0.1)

fig, ax = plt.subplots(1, figsize=(6, 6))

for i,p in enumerate(paths):
    
    data_iter = data[data['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'],data_iter['depth'], '.', color=c)
    ax.errorbar(x=data_iter['d13c'].mean(), 
                y=np.unique(data_iter['depth']), yerr=0, fmt='xr',ms=10,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, depth_dict[p]-2, p, color=c)
x = data.groupby('path')['d13c'].mean().values
y = depths
X, Y, P, rmse= line_fit(x,y)

ax.plot(X,Y, color='r', label=f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+f'RMSE={rmse:.3f}')

ax.legend()
ax.set_ylim(-180, -90)
ax.set_xlim(-3.5, -1)
ax.set_yticks(np.arange(-180, -90, 10))
ax.grid()
ax.set_ylabel('Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')

fig.savefig('depth_d13c_.pdf')

In [ ]:
#data_ = data[data['path'] != 'E']
data_ = data

depths = data_['depth'].unique()
paths = data_['path'].unique()
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap(depths/depths.min()-0.1)

fig, ax = plt.subplots(1, figsize=(6, 6))

for i,p in enumerate(paths):
    
    data_iter = data[data['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'],data_iter['depth'], '.', color=c)
    ax.errorbar(x=data_iter['d13c'].mean(), 
                y=np.unique(data_iter['depth']), yerr=0, fmt='^',color=c,ms=12,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, depth_dict[p]-2, p, color=c)
x = data_.groupby('path')['d13c'].mean().values
y = depths
X, Y, P, rmse= line_fit(x,y)

ax.plot(X,Y,'--', color='g', label='New data'+'\n'+f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+f'RMSE={rmse:.3f}')


#------------------------------------------------------------------------------------

#paths = ['A','C','D']
paths = data_old['path'].unique()
depths = data_old['depth'].unique()


viridis_cmap = plt.get_cmap('hot_r')
colors = viridis_cmap(depths/depths.min()-0.1)

for i,p in enumerate(paths):
    
    data_iter = data_old[data_old['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'], data_iter['depth'], '.', color=c)
    ax.errorbar(x= data_iter['d13c'].mean(), 
                y= data_iter['depth'].unique(), yerr=0, fmt='^',color=c,ms=12,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, path_depth[p]-2, p, color=c)

#data_iter = data_old[data_old['path'].isin(paths)]
data_iter = data_old
x = data_iter.groupby('path')['d13c'].mean().values
y = data_iter['depth'].unique()
X, Y, P, rmse= line_fit(x,y)

ax.plot(X,Y, '--', color='r', label='Old data'+'\n'+f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+f'RMSE={rmse:.3f}')

ax.legend(loc='upper left')
ax.set_ylim(-180, -60) 
ax.set_xlim(-3.5, -1)
ax.set_yticks(np.arange(-180, -50, 10))
ax.grid()
ax.set_ylabel('Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')
ax.set_title('Mean Point Fit')

fig.savefig('depth_d13c_compare.pdf')

In [ ]:
paths = ['A','B','C','D','E']
data_selected = data[data['path'].isin(paths)]
depths = data_selected['depth'].unique()

viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap(depths/depths.min()-0.1)

fig, ax = plt.subplots(1, figsize=(6, 6))

for i,p in enumerate(paths):
    
    data_iter = data[data['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'],data_iter['depth'], '.', color=c)
    ax.errorbar(x=data_iter['d13c'].mean(), 
                y=np.unique(data_iter['depth']), yerr=0, fmt='^',color=c,ms=12,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, depth_dict[p]-2, p, color=c)

data_iter = data[data['path'].isin(paths)].dropna()
x = data_iter['d13c'].values
y = data_iter['depth'].values
X, Y, P, rmse= line_fit(x,y)

ax.plot(X,Y,'--', color='g', label='New data'+'\n'+f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+f'RMSE={rmse:.3f}')


#------------------------------------------------------------------------------------

#paths = ['A','C','D']
paths = data_old['path'].unique()
depths = data_old['depth'].unique()


viridis_cmap = plt.get_cmap('hot_r')
colors = viridis_cmap(depths/depths.min()-0.1)

for i,p in enumerate(paths):
    
    data_iter = data_old[data_old['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'], data_iter['depth'], '.', color=c)
    ax.errorbar(x= data_iter['d13c'].mean(), 
                y= data_iter['depth'].unique(), yerr=0, fmt='^',color=c,ms=12,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, path_depth[p]-2, p, color=c)

#data_iter = data_old[data_old['path'].isin(paths)]
data_iter = data_old

x = data_iter['d13c'].values
y = data_iter['depth'].values
X, Y, P, rmse= line_fit(x,y)

ax.plot(X,Y, '--', color='r', label='Old data'+'\n'+f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+f'RMSE={rmse:.3f}')

ax.legend(loc='upper left')
ax.set_ylim(-180, -60) 
ax.set_xlim(-3.5, -1)
ax.set_yticks(np.arange(-180, -50, 10))
ax.grid()
ax.set_ylabel('Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')
ax.set_title('Complete Data Fit')

fig.savefig('depth_d13c_compare2.pdf')

In [ ]:
def exponential_func(x, a, b, c):
    return a * np.exp(b * x) + c

def exp_fit(X,Y, P0=np.array([1, 1, 1])):
    """returns  X_new, Y_new, popt, pcov"""
    popt, pcov = curve_fit(exponential_func, X,Y, p0=P0, maxfev=20000)
    dx = np.max(X) - np.min(X)
    X_new = np.linspace( np.min(X)-dx*0.2, np.max(X)+dx*0.2, 100)    
    Y_new = exponential_func(X_new, *popt)
    return  X_new, Y_new, popt, pcov

In [ ]:
import numpy as np
from scipy.optimize import curve_fit

def exponential_func(x, a, b, c):
    return a * np.exp(b * x) + c

def guess_initial_params(X, Y):
    X = np.asarray(X, float)
    Y = np.asarray(Y, float)
    
    # Guess c as something slightly below the minimum of Y
    c0 = Y.min() - 0.1 * abs(Y.min())
    
    # Shift Y so it's (hopefully) positive for log-transform
    Y_shifted = Y - c0
    Y_shifted[Y_shifted <= 0] = np.min(Y_shifted[Y_shifted > 0]) * 0.5
    
    # Fit log(Y - c0) ~ log(a0) + b0 * X (linear regression)
    b0, loga0 = np.polyfit(X, np.log(Y_shifted), 1)
    a0 = np.exp(loga0)
    
    return np.array([a0, b0, c0])

def exp_fit(X, Y, P0=None):
    """returns  X_new, Y_new, popt, pcov"""
    X = np.asarray(X, float)
    Y = np.asarray(Y, float)
    
    if P0 is None:
        P0 = guess_initial_params(X, Y)
    
    popt, pcov = curve_fit(
        exponential_func,
        X,
        Y,
        p0=P0,
        maxfev=20000
    )
    
    dx = np.max(X) - np.min(X)
    X_new = np.linspace(np.min(X) - dx * 0.2, np.max(X) + dx * 0.2, 100)
    Y_new = exponential_func(X_new, *popt)
    return X_new, Y_new, popt, pcov


In [ ]:
paths = ['A','B','C','D','E']
data_selected = data[data['path'].isin(paths)]
depths = data_selected['depth'].unique()

viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap(depths/depths.min()-0.1)

fig, ax = plt.subplots(1, figsize=(6, 6))

for i,p in enumerate(paths):
    
    data_iter = data[data['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'],data_iter['depth'], '.', color=c)
    ax.errorbar(x=data_iter['d13c'].mean(), 
                y=np.unique(data_iter['depth']), yerr=0, fmt='^',color=c,ms=12,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, depth_dict[p]-2, p, color=c)

data_iter = data[data['path'].isin(paths)].dropna()
x = data_iter['d13c'].values
y = data_iter['depth'].values
#X, Y, P, rmse= line_fit(x,y)

#ax.plot(X,Y,'--', color='g', label='New data'+'\n'+f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+f'RMSE={rmse:.3f}')
#X, Y, P, pcov= exp_fit(x,y, P0=np.array([50.0, 0.5, -150.0]))
X, Y, P, pcov= exp_fit(x,y)
ax.plot(X,Y, '--', color='g', label='New data')

#------------------------------------------------------------------------------------

#paths = ['A','C','D']
paths = data_old['path'].unique()
depths = data_old['depth'].unique()


viridis_cmap = plt.get_cmap('hot_r')
colors = viridis_cmap(depths/depths.min()-0.1)

for i,p in enumerate(paths):
    
    data_iter = data_old[data_old['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'], data_iter['depth'], '.', color=c)
    ax.errorbar(x= data_iter['d13c'].mean(), 
                y= data_iter['depth'].unique(), yerr=0, fmt='^',color=c,ms=12,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, path_depth[p]-2, p, color=c)

#data_iter = data_old[data_old['path'].isin(paths)]
data_iter = data_old

x = data_iter['d13c'].values
y = data_iter['depth'].values
#X, Y, P, pcov= exp_fit(x,y, P0=np.array([ 3.40573857e+04,  9.08829981e-04, -3.41132214e+04]))
X, Y, P, pcov= exp_fit(x,y)

#ax.plot(X,Y, '--', color='r', label='Old data'+'\n'+f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+f'RMSE={rmse:.3f}')
ax.plot(X,Y, '--', color='r', label='Old data')


                       
#x = np.linspace(-4,0,100)
#y=exponential_func(x , 7000,2.4,-165)
#ax.plot(x,y, '--', color='r', label='Old data')

#ax.legend(loc='upper left')
ax.set_ylim(-180, -60) 
ax.set_xlim(-4, 0)
#ax.set_yticks(np.arange(-180, -50, 10))
ax.grid()
ax.set_ylabel('Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')
ax.set_title('Complete Data Fit')

#fig.savefig('depth_d13c_compare2.pdf')

In [ ]:
data_old.to_csv('20251126_DepthLaurenPaper/data_depth.csv')

In [ ]:
fig, ax = plt.subplots(1, figsize=(8,5))

for g in ["A","D"]:
    
    ax.plot(locals()["data_"+g]["Id_num"], locals()["data_"+g].iloc[:,5],".-", label = "Depth "+g)
ax.legend()
ax.set_title(r"$\delta13C$")
fig.savefig("d13C.pdf")